# 09 Ablation Experiments

This notebook organizes the ablation analysis for the project. It runs executable structured-feature ablations using the configured ablation source and incorporates saved multimodal fusion results when those training artifacts are available.

## Ablation scope

- Vitals only
- Vitals plus labs
- Full structured EHR
- Planned text-only comparison
- Multimodal fusion comparison when Notebook 07 artifacts exist

In [11]:
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]
        if (candidate / 'configs' / 'base.yaml').exists() and (candidate / 'src').is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the project root from the current notebook directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.runtime import load_project_runtime

runtime = load_project_runtime(start=PROJECT_ROOT)
IN_COLAB = runtime.in_colab
PROJECT_ROOT = runtime.project_root
config = runtime.config
paths = runtime.paths

PROJECT_ROOT

PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy')

In [12]:
if IN_COLAB:
    %pip -q install pyyaml pandas numpy scikit-learn xgboost

In [13]:
import pandas as pd

from src.evaluation.ablation import (
    build_fusion_strategy_table,
    build_planned_ablation_matrix,
    run_structured_ablation_suite,
)
from src.utils.config import load_config
from src.utils.io_utils import save_dataframe_bundle
from src.utils.logging_utils import write_run_manifest
from src.utils.paths import ensure_directories, resolve_project_paths

In [14]:
{
    **config['ablation'],
    'tabular_multimodal_models': config.get('tabular_multimodal', {}).get('models', []),
}

{'model_source': 'tabular_multimodal',
 'executable_variants': ['vitals_only', 'vitals_labs', 'structured_full'],
 'planned_variants': ['vitals_only',
  'vitals_labs',
  'structured_full',
  'text_only',
  'multimodal_fusion'],
 'baseline_models': ['logistic_regression'],
 'tabular_multimodal_models': ['xgboost_text_augmented',
  'stacked_xgboost_notes']}

## Load horizon datasets and multimodal experiment plan

In [15]:
horizon_tables = {}
for horizon in config['prediction']['horizons_hours']:
    dataset_name = f'horizon_{int(horizon)}h'
    path = paths['processed_data_dir'] / '04_feature_engineering' / f'{dataset_name}.csv'
    assert path.exists(), f'Missing structured horizon dataset: {dataset_name}. Run 04_feature_engineering first.'
    horizon_tables[dataset_name] = pd.read_csv(path, parse_dates=['hour', 'prediction_time', 'INTIME', 'OUTTIME'], low_memory=False)

multimodal_dir = paths['processed_data_dir'] / '07_multimodal_models'
multimodal_plan_files = sorted(multimodal_dir.glob('*_multimodal_experiment_plan.csv'))
multimodal_result_files = sorted(multimodal_dir.glob('*_multimodal_results.csv'))
multimodal_plan_df = pd.concat([pd.read_csv(path) for path in multimodal_plan_files], ignore_index=True) if multimodal_plan_files else pd.DataFrame()
multimodal_results_df = pd.concat([pd.read_csv(path) for path in multimodal_result_files], ignore_index=True) if multimodal_result_files else pd.DataFrame()
{key: value.shape for key, value in horizon_tables.items()}, multimodal_plan_df.shape, multimodal_results_df.shape

({'horizon_6h': (1268294, 90),
  'horizon_12h': (1131670, 90),
  'horizon_24h': (860480, 90)},
 (12, 12),
 (24, 24))

## Run executable structured ablations

In [16]:
artifacts = run_structured_ablation_suite(horizon_tables, config)
ablation_results_df = artifacts['ablation_results']
ablation_results_df

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/home/sra/shankari/Multimodal-Sepsis-Prediction-main/.venv/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (st

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/home/sra/shankari/Multimodal-Sepsis-Prediction-main/.venv/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (st

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,dataset_name,split,model_name,text_embedding_backend,decision_threshold,n_features,n_structured_features,n_text_embedding_features,n_note_metadata_features,n_event_features,...,f1,brier_score,specificity,sensitivity,tp,fp,tn,fn,stack_train_examples,variant_name
0,horizon_12h,test,stacked_xgboost_notes,transformer:emilyalsentzer/Bio_ClinicalBERT,0.500000,25,428,1536,14,9,...,0.935743,0.006624,0.995083,0.974895,233,26,5262,6,5431.0,structured_full
1,horizon_12h,test,xgboost_text_augmented,transformer:emilyalsentzer/Bio_ClinicalBERT,0.250000,1987,428,1536,14,9,...,0.935484,0.004451,0.995272,0.970711,232,25,5263,7,NaN,structured_full
2,horizon_12h,val,xgboost_text_augmented,transformer:emilyalsentzer/Bio_ClinicalBERT,0.250000,1987,428,1536,14,9,...,0.944578,0.003855,0.996940,0.965517,196,16,5212,7,NaN,structured_full
3,horizon_12h,test,stacked_xgboost_notes,transformer:emilyalsentzer/Bio_ClinicalBERT,0.500000,16,360,1536,14,0,...,0.221729,0.170572,0.817700,0.627615,150,964,4324,89,5431.0,vitals_labs
4,horizon_12h,test,xgboost_text_augmented,transformer:emilyalsentzer/Bio_ClinicalBERT,0.072436,1910,360,1536,14,0,...,0.391198,0.035497,0.982980,0.334728,80,90,5198,159,NaN,vitals_labs
5,horizon_12h,val,xgboost_text_augmented,transformer:emilyalsentzer/Bio_ClinicalBERT,0.072436,1910,360,1536,14,0,...,0.331551,0.031966,0.979151,0.305419,62,109,5119,141,NaN,vitals_labs
6,horizon_12h,test,stacked_xgboost_notes,transformer:emilyalsentzer/Bio_ClinicalBERT,0.500000,16,160,1536,14,0,...,0.175764,0.207320,0.729198,0.673640,161,1432,3856,78,5431.0,vitals_only
7,horizon_12h,test,xgboost_text_augmented,transformer:emilyalsentzer/Bio_ClinicalBERT,0.070273,1710,160,1536,14,0,...,0.268585,0.039352,0.976929,0.234310,56,122,5166,183,NaN,vitals_only
8,horizon_12h,val,xgboost_text_augmented,transformer:emilyalsentzer/Bio_ClinicalBERT,0.070273,1710,160,1536,14,0,...,0.203125,0.034965,0.972839,0.192118,39,142,5086,164,NaN,vitals_only
9,horizon_24h,test,stacked_xgboost_notes,transformer:emilyalsentzer/Bio_ClinicalBERT,0.500000,25,428,1536,14,9,...,0.953168,0.006909,0.996097,0.988571,173,15,3828,2,3997.0,structured_full


## Build planned ablation matrix and fusion strategy table

In [17]:
planned_matrix_df = build_planned_ablation_matrix(config)
fusion_source_df = multimodal_results_df if not multimodal_results_df.empty else multimodal_plan_df
fusion_strategy_df = build_fusion_strategy_table(fusion_source_df)
planned_matrix_df, fusion_strategy_df

(        variant_name                                        description  \
 0        vitals_only                      Structured vitals subset only   
 1        vitals_labs                      Vitals plus laboratory subset   
 2    structured_full                        All structured EHR features   
 3          text_only         Clinical notes without structured features   
 4  multimodal_fusion  Structured plus text with multiple fusion stra...   
 
    implemented_now  
 0             True  
 1             True  
 2             True  
 3            False  
 4            False  ,
    structured_encoder dataset_name split     auprc     auroc      loss  \
 0         transformer  horizon_12h  test  0.082739  0.636393  0.180825   
 1         transformer  horizon_12h  test  0.091342  0.660434  0.189863   
 2         transformer  horizon_12h  test  0.090082  0.650558  0.231950   
 3         transformer  horizon_12h  test  0.100267  0.670186  0.176459   
 4         transformer  horizon_12

## Inspect strongest ablation results

In [18]:
if ablation_results_df.empty:
    ablation_results_df
else:
    review_df = ablation_results_df.copy()
    if 'split' in review_df.columns:
        review_df = review_df.loc[review_df['split'] == 'test'].copy()
    summary_columns = [
        column
        for column in [
            'dataset_name',
            'variant_name',
            'model_name',
            'split',
            'auroc',
            'auprc',
            'accuracy',
            'precision',
            'recall',
            'f1',
            'decision_threshold',
            'n_features',
            'n_structured_features',
            'n_text_embedding_features',
            'n_note_metadata_features',
            'n_event_features',
        ]
        if column in review_df.columns
    ]
    review_df.loc[:, summary_columns].sort_values(['dataset_name', 'auprc'], ascending=[True, False]).head(18)


## Save ablation artifacts

In [19]:
output_dir = paths['processed_data_dir'] / '09_ablation_experiments'
artifact_bundle = dict(artifacts)
artifact_bundle['planned_ablation_matrix'] = planned_matrix_df
artifact_bundle['fusion_strategy_table'] = fusion_strategy_df
saved_paths = save_dataframe_bundle(artifact_bundle, output_dir)
saved_paths

{'horizon_6h_vitals_only_xgboost_text_augmented_val_predictions': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/09_ablation_experiments/horizon_6h_vitals_only_xgboost_text_augmented_val_predictions.csv',
 'horizon_6h_vitals_only_xgboost_text_augmented_test_predictions': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/09_ablation_experiments/horizon_6h_vitals_only_xgboost_text_augmented_test_predictions.csv',
 'horizon_6h_vitals_only_stacked_xgboost_notes_test_predictions': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/09_ablation_experiments/horizon_6h_vitals_only_stacked_xgboost_notes_test_predictions.csv',
 'horizon_6h_vitals_only_tabular_multimodal_feature_manifest': '/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/processed/09_ablation_experiments/horizon_6h_vitals_only_tabular_multimodal_feature_manifest.csv',
 'horizon_6h_vitals_only_tabular_multimodal_results': '/home/sra

In [20]:
manifest_path = paths['manifests_dir'] / '09_ablation_experiments_manifest.json'
write_run_manifest(
    path=manifest_path,
    stage='09_ablation_experiments',
    config=config,
    extra={
        'saved_artifacts': saved_paths,
        'multimodal_plan_available': bool(len(multimodal_plan_df) or len(multimodal_results_df)),
        'ablation_result_rows': int(len(ablation_results_df)),
    },
)
manifest_path

PosixPath('/home/sra/shankari/Multimodal-Sepsis-Prediction-main-copy/results/manifests/09_ablation_experiments_manifest.json')

## Review checklist

Before explainability analysis, review:

- How much performance comes from vitals alone?
- Do labs add meaningful signal beyond vitals?
- Which horizons benefit most from richer structured features?
- Which fusion strategies should be prioritized once full multimodal training outputs are available?
- Is the planned ablation matrix complete enough for the paper?

## Next notebook

`10_explainability.ipynb` will prepare SHAP-based structured explanations, attention-based note interpretation, and clinically meaningful narratives for the final report.